In [30]:
pip install jupyter ipykernel numpy pandas matplotlib tqdm vector tables scipy


Note: you may need to restart the kernel to use updated packages.


In [54]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import os
import pickle

In [47]:

# Update the file path to your local file
local_file_path = r"C:\Users\HB\OneDrive\Desktop\OmniFold\.ipynb_checkpoints

if os.path.isfile(local_file_path):
    print("Skipping download -- data already exists locally.")
else:
    print("File not found at the specified path.")

try:
    from multifold_util import *
    import numpy as np
    import pandas as pd
    pd.options.mode.chained_assignment = None
    import matplotlib.pyplot as plt
    from tqdm import tqdm
    import os
    import pickle

    if os.path.isfile(local_file_path):
        print("Done!")
    else:
        raise Exception("File does not exist.")

except Exception as e:
    print(f"There was an error: {e}")

Skipping download -- data already exists locally.
There was an error: No module named 'multifold_util'


In [48]:
### Default: load the full measurement
n_events = None


In [52]:
print("Loading the measurement - based on actual data.")
#folder = os.path.join(top_folder, "data")

# Nominal measurement with most uncertainties (based on MG5-Drell Yan)
# Fixed: Using raw string (r"") to prevent backslashes from being interpreted as escape characters
multifold = pd.read_hdf(r"C:\Users\HB\Downloads\.ipynb_checkpoints\files\pseudodata\multifold.h5", stop = n_events)

# Sherpa-based, alternative measurement. Difference with nominal is considered as an uncertainty.
multifold_sherpa = pd.read_hdf(r"C:\Users\HB\Downloads\.ipynb_checkpoints\files\pseudodata\multifold_sherpa.h5", stop = n_events) # REMOVED the 0.8 is to make the nEff approximately equal to the ratio in the full measurement

# Alternative measurement that includes EW Zjj (i.e. 'VBF') and diboson components in the unfolding. Difference with nominal is considered another uncertainty. 
multifold_nonDY = pd.read_hdf(r"C:\Users\HB\Downloads\.ipynb_checkpoints\files\pseudodata\multifold_nonDY.h5", stop = n_events)

Loading the measurement - based on actual data.


In [53]:
multifold

,pT_ll,pT_l1,pT_l2,eta_l1,eta_l2,phi_l1,phi_l2,y_ll,pT_trackj1,y_trackj1,...,weights_trackPtScale,weights_theoryPSjet,weights_theoryPSsoft,weights_theoryMPI,weights_theoryPSscale,weights_theoryAlphaS,weights_theoryQCD,weights_theoryPDF,weights_topBackground,weights_lumi
0,479.442780,288.466919,198.183929,-0.117443,-0.334414,0.893844,0.543039,-0.205878,243.584351,-0.924242,...,0.003318,0.002974,0.003341,0.003439,0.003302,0.003139,0.003394,0.003297,0.003390,0.003577
1,274.524994,166.120789,125.378044,0.313321,0.368928,2.230892,1.537930,0.337239,96.407021,-1.158080,...,0.003974,0.004168,0.004008,0.003964,0.004520,0.004160,0.003957,0.004212,0.003988,0.003921
2,462.713226,335.697479,133.157684,0.766387,0.534180,1.191890,0.832184,0.700256,86.603012,-1.048886,...,0.002270,0.002390,0.002620,0.002389,0.002341,0.002371,0.001995,0.002238,0.002421,0.002502
3,215.157608,189.518021,25.711994,1.083798,-0.190828,2.326127,2.406116,0.904586,222.285919,0.166012,...,0.006548,0.006485,0.006521,0.006400,0.006396,0.006566,0.007135,0.006141,0.006462,0.006676
4,222.458313,128.850159,108.589226,-0.635713,-0.972127,-2.909612,2.656590,-0.789699,206.385544,0.392547,...,0.002490,0.002863,0.002522,0.002721,0.002586,0.002664,0.002733,0.002729,0.002723,0.002674
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
418009,934.971924,738.464722,196.525192,0.102944,-0.133273,2.388935,2.404151,0.053082,475.055756,0.811634,...,0.000030,0.000031,0.000027,0.000032,0.000035,0.000028,0.000026,0.000033,0.000028,0.000030
418010,245.813461,166.847061,93.757919,1.308837,1.533588,1.591105,0.884510,1.389817,713.371887,1.452970,...,0.000229,0.000224,0.000241,0.000235,0.000262,0.000226,0.000257,0.000228,0.000244,0.000233
418011,478.670349,378.737518,108.016479,-0.328871,-0.227935,-1.235470,-1.675783,-0.306456,100.177452,0.050614,...,0.002307,0.002124,0.002138,0.002180,0.002257,0.002170,0.002150,0.002294,0.002219,0.002249
418012,278.586029,249.255356,43.581135,0.632484,0.687496,-0.966683,-0.071259,0.640673,224.825485,1.490560,...,0.004303,0.004123,0.004108,0.004109,0.004163,0.003991,0.002836,0.004381,0.003993,0.004208


In [55]:
def printEventFeatures(dataset,desc):
    columns = np.array(dataset.keys())
    NmcBS=NdataBS=Nsys=Nens=Nvars=0
    for column in columns:
        if "bootstrap_mc" in column:
            NmcBS+=1
        elif "bootstrap_data" in column:
            NdataBS+=1
        elif "weights_ensemble" in column:
            Nens+=1
        elif "weight" in column:
            Nsys+=1
        elif column!="target_dd": # avoid couting special varialbe for DD unfold syst
            Nvars+=1
    print("\nFor {}, there are {} event properties:".format(desc,len(columns)))
    print("  {} variables, {} weights, {} NN ensemble weights, {} MC and {} data bootstrap weights".format(Nvars,Nsys,Nens,NmcBS,NdataBS))

printEventFeatures(multifold,"the main dataset")
printEventFeatures(multifold_sherpa,"the alternative, Sherpa-derived measurement")
printEventFeatures(multifold_nonDY,"the dataset derived with non-Drell-Yan components")


For the main dataset, there are 200 event properties:
  24 variables, 25 weights, 100 NN ensemble weights, 25 MC and 25 data bootstrap weights

For the alternative, Sherpa-derived measurement, there are 51 event properties:
  24 variables, 2 weights, 0 NN ensemble weights, 25 MC and 0 data bootstrap weights

For the dataset derived with non-Drell-Yan components, there are 26 event properties:
  24 variables, 2 weights, 0 NN ensemble weights, 0 MC and 0 data bootstrap weights


In [65]:
# Method to obtain measured and predicted cross section of an event sample corresponding
# to a (fiducial) kinematic subregion
def printCrossSection(dataset,desc,wnom_name="weights_nominal"):
    meas_xsec = np.sum(dataset[wnom_name])
    pred_xsec = np.sum(dataset.weight_mc)
    sumw2 = np.sum(dataset[wnom_name]**2)
    nEff = meas_xsec*meas_xsec/sumw2 # => see nEff method
    print("\nFor {}, we have:".format(desc))
    print("  Measured x-sec:     {:.2f} fb".format(meas_xsec))
    print("  MC-predicted x-sec: {:.2f} fb".format(pred_xsec))
    # Fixed: Use nEff directly as a variable, not as a function call
    print("  Effective statistics: {:.1f}".format(nEff))

printCrossSection(multifold,"nominal, MG5-based result, full fiducial phase space")
printCrossSection(multifold_sherpa,"Sherpa-based result, full fiducial phase space")
printCrossSection(multifold_nonDY,"Non-DY, full fiducial phase space")

# Define a fiducial subset ("A" in equations above)
print("\n\n===== FIDUCIAL VOLUME pT(ll) > 500 GeV =====")

meas_pT500 = multifold[multifold.pT_ll > 500]
sherpa_pT500 = multifold_sherpa[multifold_sherpa.pT_ll > 500]
nonDY_pT500 = multifold_nonDY[multifold_nonDY.pT_ll > 500]

printCrossSection(meas_pT500,"nominal, MG5-based result, in region pT(ll) > 500 GeV")
printCrossSection(sherpa_pT500,"Sherpa-based result, in region pT(ll) > 500 GeV")
printCrossSection(nonDY_pT500,"Non-Drell-Yan result, in region pT(ll) > 500 GeV")

#ASK WHY VALUES LOOK DIFFERENT HERE AND IN UNCERTAINTIES


For nominal, MG5-based result, full fiducial phase space, we have:
  Measured x-sec:     1809.46 fb
  MC-predicted x-sec: 1701.13 fb
  Effective statistics: 213024.2

For Sherpa-based result, full fiducial phase space, we have:
  Measured x-sec:     1816.17 fb
  MC-predicted x-sec: 1601.73 fb
  Effective statistics: 199001.4

For Non-DY, full fiducial phase space, we have:
  Measured x-sec:     1810.59 fb
  MC-predicted x-sec: 1780.72 fb
  Effective statistics: 221387.3


===== FIDUCIAL VOLUME pT(ll) > 500 GeV =====

For nominal, MG5-based result, in region pT(ll) > 500 GeV, we have:
  Measured x-sec:     41.48 fb
  MC-predicted x-sec: 39.47 fb
  Effective statistics: 25127.2

For Sherpa-based result, in region pT(ll) > 500 GeV, we have:
  Measured x-sec:     41.58 fb
  MC-predicted x-sec: 34.93 fb
  Effective statistics: 15509.1

For Non-Drell-Yan result, in region pT(ll) > 500 GeV, we have:
  Measured x-sec:     41.76 fb
  MC-predicted x-sec: 43.43 fb
  Effective statistics: 21523.6

In [66]:
# Lists of uncertainty variations
syst_nps = ["pileup","lumi","topBackground", # pileup modelling, luminosity, bkg subtraction (top). 3 NPs
            "trackEffMain","trackEffJet","trackFake","trackPtScale", # track systematics. 4 NPs
            "muEffReco","muEffIso","muEffTrack","muEffTrig","muCalID","muCalMS","muCalResBias","muCalScale", # muon eff+calib. 8 NPs
            "theoryPSjet","theoryPSsoft","theoryMPI","theoryPSscale","theoryAlphaS","theoryQCD","theoryPDF"] # theory. 7 NPs

mc_train_stat  = [col for col in multifold.keys() if col.startswith("weights_bootstrap_mc")]
data_stat      = [col for col in multifold.keys() if col.startswith("weights_bootstrap_data")]
NN_stability   = [col for col in multifold.keys() if col.startswith("weights_ensemble")]

def printXsecUnc(dataset,desc,sherpa,nonDY):
    meas_xsec  = np.sum(dataset["weights_nominal"])    # sum w
    Vstat_meas = np.sum(dataset["weights_nominal"]**2) # sum w^2, statisical variance

    Vmeas = 0 # total, fractional variance
    print("\n---------\n{}".format(desc))
    print("   Measured x-sec:   {:.2f} fb".format(meas_xsec))
    print("\n   Systematics:")
    for NP in syst_nps:
        syst_xsec = np.sum(dataset["weights_"+NP])
        unc = (syst_xsec/meas_xsec-1) # fractional uncertainty amplitude
        Vmeas += unc*unc
        print("  {:>15s}:    {:5.2f}%".format(NP,unc*100))
        
    # Measurments performed with alternative MC samples 
    meas_sherpa = np.sum(sherpa["weights_nominal"]) # 'hidden variable uncertainty'
    meas_nonDY  = np.sum(nonDY["weights_nominal"]) # EW Zjj + diboson modelling 
    meas_DD  = np.sum(dataset["weights_dd"]) # Data-driven unfolding uncertainty
    meas_DD_target  = np.sum(dataset["target_dd"])
    print("  {:>15s}:    {:5.2f}%".format("DD unfold. unc.",(meas_DD/meas_DD_target-1)*100))
    
    Vmeas += (meas_DD/meas_DD_target-1)**2
    Vmeas += (meas_sherpa/meas_xsec-1)**2
    Vmeas += (meas_nonDY/meas_xsec-1)**2
    print("  {:>15s}:    {:5.2f}% (HV unfold. unc.)".format("Sherpa meas",(meas_sherpa/meas_xsec-1)*100))
    print("  {:>15s}:    {:5.2f}%".format("Non-DY meas",(meas_nonDY/meas_xsec-1)*100))
    print("  {:>15s}:    {:5.2f}%".format("Total syst",Vmeas**0.5*100))

    print("\n   Stochastic uncertainties:")
    print("  {:>15s}:    {:5.2f}%".format("MC stat",Vstat_meas**0.5/meas_xsec*100))
    Vmeas += Vstat_meas/meas_xsec/meas_xsec
    
    unc = np.std([np.sum(dataset[col]) for col in mc_train_stat])/meas_xsec
    Vmeas += unc*unc
    print("  {:>15s}:    {:5.2f}%".format("MC train stat",unc*100))

    unc = np.std([np.sum(dataset[col]) for col in data_stat])/meas_xsec
    Vmeas += unc*unc
    print("  {:>15s}:    {:5.2f}%".format("Data stat",unc*100))

    unc = np.std([np.sum(dataset[col]) for col in NN_stability])/meas_xsec
    Vmeas += unc*unc
    print("  {:>15s}:    {:5.2f}%".format("NN stability",unc*100))
    print("\n  {:>15s}:    {:5.2f}%".format("Total uncert",Vmeas**0.5*100))

In [67]:
printXsecUnc(multifold,"Full fiducial phase space, i.e. pT(ll) > 200 GeV",multifold_sherpa,multifold_nonDY)


---------
Full fiducial phase space, i.e. pT(ll) > 200 GeV
   Measured x-sec:   1809.46 fb

   Systematics:
           pileup:    -0.10%
             lumi:     1.70%
    topBackground:    -0.19%
     trackEffMain:    -0.04%
      trackEffJet:    -0.01%
        trackFake:    -0.02%
     trackPtScale:    -0.03%
        muEffReco:     0.56%
         muEffIso:     1.16%
       muEffTrack:     0.28%
        muEffTrig:     0.28%
          muCalID:    -0.11%
          muCalMS:    -0.08%
     muCalResBias:     0.02%
       muCalScale:    -0.27%
      theoryPSjet:    -0.00%
     theoryPSsoft:    -0.00%
        theoryMPI:    -0.01%
    theoryPSscale:    -0.09%
     theoryAlphaS:     0.13%
        theoryQCD:     0.01%
        theoryPDF:    -0.01%
  DD unfold. unc.:     0.80%
      Sherpa meas:     0.37% (HV unfold. unc.)
      Non-DY meas:     0.06%
       Total syst:     2.38%

   Stochastic uncertainties:
          MC stat:     0.22%
    MC train stat:     0.10%
        Data stat:     0.21%
  

In [68]:
printXsecUnc(meas_pT500,"pT(ll) > 500 GeV",sherpa_pT500,nonDY_pT500)


---------
pT(ll) > 500 GeV
   Measured x-sec:   41.48 fb

   Systematics:
           pileup:    -0.23%
             lumi:     1.70%
    topBackground:     1.22%
     trackEffMain:     1.07%
      trackEffJet:    -0.76%
        trackFake:    -0.70%
     trackPtScale:    -1.26%
        muEffReco:    -0.31%
         muEffIso:     1.48%
       muEffTrack:    -0.71%
        muEffTrig:     0.79%
          muCalID:     0.29%
          muCalMS:     0.17%
     muCalResBias:    -0.44%
       muCalScale:    -0.13%
      theoryPSjet:     0.12%
     theoryPSsoft:     0.17%
        theoryMPI:    -0.59%
    theoryPSscale:     1.43%
     theoryAlphaS:     0.55%
        theoryQCD:     0.06%
        theoryPDF:     0.17%
  DD unfold. unc.:     2.51%
      Sherpa meas:     0.26% (HV unfold. unc.)
      Non-DY meas:     0.70%
       Total syst:     4.64%

   Stochastic uncertainties:
          MC stat:     0.63%
    MC train stat:     0.63%
        Data stat:     1.49%
     NN stability:     7.50%

     T